# Assessment Notebook

## Problem 1: Representing SHA-256 Data

This porblem involves: research implementing SHA-256, investigate how the input, output, and intermediate data can be represented in Python.

### 32-bit words

SHA-256 operates on groups of 32 bits called words. Each word is an unsigned integer with a range from (0) to (2^{32} - 1). A 32-bit word can be represented using eight hexadecimal digits, as each hexadecimal digit represents four bits.

For example:

In [1]:
import numpy as np

word = np.uint32(0x12345678)

print(word)
print(word.dtype)

305419896
uint32


`numpy.uint32` is appropriate because it explicitly represents an unsigned 32-bit integer, matching the word size used by SHA-256. This is preferable to a floating-point type because SHA-256 relies on integer and bitwise operations such as XOR, AND and bit shifting.

The fixed 32-bit representation is also important because SHA-256 uses arithmetic modulo (2^{32}). For example:

In [2]:
import numpy as np

word = np.uint32(0xFFFFFFFF)
one = np.uint32(1)

result = np.uint32((int(word) + int(one)) & 0xFFFFFFFF)

print(f"{result:08x}")

00000000


This results in `00000000`, demonstrating the 32-bit wraparound from (2^{32} - 1 + 1).

Hexadecimal is mainly used as a convenient way of displaying the word; the underlying value is still a 32-bit unsigned integer.

### Sequences of 32-bit words

A SHA-256 word is exactly 32 bits, so it can represent an unsigned integer in the range:

$$
0 \leq x \leq 2^{32} - 1
$$

In Python, a sequence of these words can be represented using a NumPy array with the `numpy.uint32` data type. This ensures that each element occupies exactly 32 bits and that arithmetic and bitwise operations behave similarly to the operations required by SHA-256.

For example:

In [3]:
import numpy as np

words = np.array([
    0x12345678,
    0xABCDEF01,
    0x00000000,
    0xFFFFFFFF
], dtype=np.uint32)

print(words)
print(words.dtype)

[ 305419896 2882400001          0 4294967295]
uint32


This produces an array where every element is explicitly represented as a 32-bit unsigned integer.

The hexadecimal notation is useful because eight hexadecimal digits correspond to 32 bits:
$$
0x12345678=00010010 00110100 01010110 01111000
$$
A NumPy `uint32` is particularly suitable for SHA-256 because the algorithm requires operations such as addition modulo $2^{32}$, bitwise AND, OR, XOR, and rotations.

### Input Messages

SHA-256 accepts an input message of arbitrary length. The message is ultimately processed as a sequence of bytes, so bytes is an appropriate representation in Python.

A text message can initially be represented as a Python `str`, but it must be converted into bytes before being passed to SHA-256. This can be done using the `.encode()` method. UTF-8 encoding is suitable for representing text, with each character being converted into one or more bytes.

For example, the message `"abc"` can be represented as follows:

In [4]:
# Example input message 
message = "abc" 

# Convert the message into UTF-8 bytes 
message_bytes = message.encode("utf-8") 

print("Message:", message) 
print("Bytes:", message_bytes) 
print("Length:", len(message_bytes))

Message: abc
Bytes: b'abc'
Length: 3


The resulting `bytes` object contains the three byte values corresponding to the characters `a`, `b`, and `c`. These bytes form the input data that can then be processed by the SHA-256 algorithm.

The message `"abc"` is also used as an example in the SHA-256 specification ([FIPS 180-4, Section 5.1.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=18)).

### 512-bit message blocks

SHA-256 processes messages in blocks of 512 bits, equivalent to 64 bytes. According to ([NIST FIPS 180-4, Section 5.2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=17)), the padded message is divided into 512-bit blocks, with each block consisting of sixteen 32-bit words.

In Python, a block can be represented using a `bytes` object containing exactly 64 bytes. This is appropriate because each byte represents eight bits, making it straightforward to store and manipulate the binary input. The bytes can then be converted into a NumPy array of sixteen `uint32` values, matching the word structure used by SHA-256.

For example:

In [9]:
import numpy as np

# Create an example 512-bit message block
block = bytes(range(64))

print("Block size in bytes:", len(block))
print("Block size in bits:", len(block) * 8)
print("First 8 bytes:", block[:8])

Block size in bytes: 64
Block size in bits: 512
First 8 bytes: b'\x00\x01\x02\x03\x04\x05\x06\x07'


The block contains 64 bytes, giving a total of 512 bits. The `bytes` representation is useful for storing the original message data, while NumPy can be used to represent the sixteen 32-bit words required for processing.

Each group of four bytes must be interpreted in big-endian order, meaning that the most significant byte comes first. This follows the convention specified in FIPS 180-4.

In [10]:
# Convert each group of four bytes into a 32-bit word
words = np.array([
    int.from_bytes(block[i:i + 4], byteorder="big")
    for i in range(0, 64, 4)
], dtype=np.uint32)

print("Number of words:", len(words))
print("Data type:", words.dtype)
print("First four words:", [f"{word:08x}" for word in words[:4]])

Number of words: 16
Data type: uint32
First four words: ['00010203', '04050607', '08090a0b', '0c0d0e0f']


This produces sixteen unsigned 32-bit integers. For this example, the first four words are 00010203, 04050607, 08090a0b, and 0c0d0e0f.

Using `numpy.uint32` is appropriate because SHA-256 performs its operations on 32-bit words, including bitwise operations and addition modulo $(2^{32})$. The array representation therefore provides a convenient way to organise the message block for subsequent stages of the algorithm.

The original bytes object and the array of words serve different purposes: bytes preserves the input as a sequence of eight-bit values, while the NumPy array provides the sixteen 32-bit words that the SHA-256 compression function uses.

### The final 256-bit hash value

SHA-256 produces a fixed-length message digest of 256 bits, regardless of the length of the input message. The digest is the final output of the hashing algorithm and provides a compact representation of the input data. NIST specifies the digest size and algorithm in the Secure Hash Standard ([FIPS 180-4, Section 6.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)).

Since one byte contains eight bits, the digest occupies:

$$
\frac{256\text{ bits}}{8\text{ bits per byte}} = 32\text{ bytes}
$$

A SHA-256 digest is commonly displayed in hexadecimal form. Each hexadecimal digit represents four bits, so the complete digest contains:

$$
\frac{256\text{ bits}}{4\text{ bits per hexadecimal digit}} = 64\text{ hexadecimal digits}
$$

In Python, the digest can be represented as a bytes object containing 32 bytes and converted into a hexadecimal string using the .hex() method. This is appropriate because bytes stores the digest as a sequence of eight-bit values, while hexadecimal provides a readable representation of the same binary data. Each byte requires two hexadecimal digits, so the hexadecimal representation is twice as long as the byte sequence.

For example, the SHA-256 test message "abc" produces the following digest. This value is defined by the SHA-256 specification and is a 256-bit output, as described in ([NIST FIPS 180-4, Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)).

In [1]:
# Example final SHA-256 hash value for "abc"
hash_value = bytes.fromhex(
    "ba7816bf8f01cfea414140de5dae2223"
    "b00361a396177a9cb410ff61f20015ad"
)

print("Hash:", hash_value.hex())
print("Size in bytes:", len(hash_value))
print("Size in bits:", len(hash_value) * 8)
print("Hexadecimal digits:", len(hash_value.hex()))

Hash: ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
Size in bytes: 32
Size in bits: 256
Hexadecimal digits: 64


The code uses `bytes.fromhex()` to convert the hexadecimal representation into its corresponding sequence of bytes. It does not calculate the digest; instead, it stores a known test value so that its representation and size can be examined without using a hashing library.

Internally, SHA-256 maintains its hash value as eight 32-bit words. After all message blocks have been processed, these words are concatenated in order to form the final 256-bit digest. The total size is therefore:

$$
8 \times 32 = 256\text{ bits}
$$

This connects the final digest representation to the intermediate data structures used by the algorithm.

Therefore, a `bytes` object is suitable for representing the complete digest, a hexadecimal string is useful for displaying it, and eight 32-bit words describe its structure within the SHA-256 algorithm.

## Problem 2: SHA-256 Bitwise Operations

This problem involves: 

## Problem 3: Generating the SHA-256 Constants

This problem involves: 

## Problem 4: Padding and Parsing Messages

This problem involves: 

## Problem 5: The SHA-256 Compression Function

This problem involves: 

## Problem 6: Complete SHA-256

This problem involves: 

## End